# Solutions — Wanderbricks Reviews (Medium 06)

**Dataset:** `samples.wanderbricks.reviews`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

reviews = spark.table("samples.wanderbricks.reviews")

## Solution 1 — Avg Rating per Property (Non-Deleted Only)

In [ ]:
result_1 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .groupBy("property_id")
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("*").alias("review_count")
    )
    .orderBy(F.col("review_count").desc())
)

## Solution 2 — Properties with Perfect 5.0 Average Rating

In [ ]:
result_2 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .groupBy("property_id")
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("*").alias("review_count")
    )
    .filter(F.col("avg_rating") == 5.0)
    .orderBy(F.col("review_count").desc())
)

## Solution 3 — Rating Distribution with Percentages

In [ ]:
total = reviews.filter(F.col("is_deleted") == False).count()
result_3 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .groupBy("rating")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(F.col("count") / total * 100, 1))
    .orderBy("rating")
)

## Solution 4 — Review Count per Month

In [ ]:
result_4 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .withColumn("year", F.year("created_at"))
    .withColumn("month", F.month("created_at"))
    .groupBy("year","month")
    .agg(F.count("*").alias("review_count"))
    .orderBy("year","month")
)

## Solution 5 — Top 10 Properties by Review Count with Avg Rating

In [ ]:
result_5 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .groupBy("property_id")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
    .orderBy(F.col("review_count").desc())
    .limit(10)
)

## Solution 6 — Running Average Rating per Property

In [ ]:
# Why: partitionBy property ordered by created_at gives per-property rolling avg
w = Window.partitionBy("property_id").orderBy("created_at").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result_6 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .select(
        "review_id","property_id","rating","created_at",
        F.round(F.avg("rating").over(w), 2).alias("running_avg_rating")
    )
    .orderBy("property_id","created_at")
)

## Solution 7 — Properties Where Latest Review Rating < 3.0

In [ ]:
# Why: row_number per property desc by date lets us grab the most recent review
w = Window.partitionBy("property_id").orderBy(F.col("created_at").desc())
result_7 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .filter(F.col("rating") < 3.0)
    .select(
        "property_id",
        F.to_date("created_at").alias("latest_review_date"),
        F.col("rating").alias("latest_rating")
    )
    .orderBy("latest_rating")
)